# Consistency: retraining on rubric ratings only

**What changed from the first clean run.** The first run had 20 rated records; the labels of the other 124 real records and of the 175
simulated records were estimates from a model fitted on those 20. Here the statistical propagation is gone: every real and every
simulated record has a rubric rating (`dataset/reference_ratings/`), the factor's label is the mean of its criteria rescaled to 0-1,
and the models are trained and tested on those ratings. The protocol is the same ([RETRAIN_PROTOCOL.md](../../RETRAIN_PROTOCOL.md),
amendment log); the success criteria were not touched.

**Who rated, plainly.** The 20 protocol ratings and the 299 new ones were all written by the assistant against the rubric. No person rated
anything, and the simulated students' ratings share an author with their data. A good result below shows that the models reproduce the
rubric as the assistant applies it. It is **not** human validation and not independent ground truth.

**Evaluation.** Primary: pooled over all 144 real records, each predicted by a model that never saw that student's group. Secondary: the
original 20 records only, for comparison with the first run. Simulated students are training only.

In [ ]:
import json
import os
import sys
import warnings
from pathlib import Path

os.environ["OMP_NUM_THREADS"] = "1"
sys.path.insert(0, str(Path("../../ml_lib").resolve()))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import retrain as R
from core import GROUPS, group_bootstrap, load, make_splits, md_table, metrics, paired_diff_bootstrap
from retrain_configs import CONFIGS

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

cfg = CONFIGS["consistency_ref"]()
F = cfg.factor()
RES = F.results
d = load(F)
real, sim = d[d.source == "real"], d[d.source == "synthetic"]
gold = real[real.label_source == "rater"]
P20 = set(real[real.rating_source == "protocol-20"].record_id)
LABEL, ACT = cfg.label, cfg.activity
print(cfg.title, "| real", len(real), "all rated (protocol-20:", len(P20), "+ reference:", len(real) - len(P20), ") | simulated", len(sim), "training only, own ratings")

## 1. The labels

In [ ]:
rows = []
for name, frame in (("real, protocol-20", real[real.rating_source == "protocol-20"]), ("real, reference", real[real.rating_source == "reference"]),
                    ("simulated", sim)):
    y = frame[LABEL].astype(float)
    rows.append({"records": name, "n": len(y), "mean": y.mean(), "sd": y.std(), "p10": y.quantile(.1), "median": y.median(), "p90": y.quantile(.9)})
display(pd.DataFrame(rows).set_index("records").round(3))
first = pd.read_csv(Path("../clean/dataset") / cfg.dataset_file, low_memory=False).set_index("record_id")
prop = first[(first.source == "real") & (first.label_source == "propagated")]
both = prop.label.astype(float).to_frame("propagated (first run)").join(real[LABEL].rename("rubric rating"))
print("The 124 real records whose label was an estimate in the first run: estimate against rubric rating")
print("  correlation %.3f | mean abs difference %.3f" % (both.corr().iloc[0, 1], (both.iloc[:, 0] - both.iloc[:, 1]).abs().mean()))
fig, ax = plt.subplots(figsize=(4.6, 4.4))
ax.scatter(both["rubric rating"], both["propagated (first run)"], s=18, c="#2b6cb0")
ax.plot([0, 1], [0, 1], color="grey", lw=1); ax.set_xlabel("rubric rating"); ax.set_ylabel("first-run estimate"); fig.tight_layout(); plt.show()

## 2. Splits and baselines (validation)

In [ ]:
splits = make_splits(d)
splits.to_csv(F.splits, index=False)
role = splits.join(d[["source", "label_source"]], on="record_id")
chk = role.groupby(["outer_test", "inner_val", "role"]).size().unstack(fill_value=0)
assert not ((role.role == "val") & (role.source == "synthetic")).any() and not ((role.role == "test") & (role.source == "synthetic")).any()
print("20 combinations; simulated rows are never in validation or test. Records per role (first rows):")
display(chk.head(8))
base = pd.read_csv(RES / "baseline_predictions.csv")
rows = []
for name, g in base.groupby("name"):
    m = metrics(g.y, g.pred)
    rows.append({"baseline": name, "MAE": m["MAE"], "pearson": m["pearson"], "spearman": m["spearman"], "bias": m["bias"]})
display(pd.DataFrame(rows).set_index("baseline").sort_values("MAE").round(3))

## 3. Preprocessing: variants and label sources (Ridge, validation, selection without bias)

In [ ]:
tuned = pd.read_csv(RES / "tuning_predictions.csv")
settings = json.load(open(RES / "settings.json"))
rows = []
for (variant, lab, weight), g in tuned[tuned.model == "Ridge"].groupby(["variant", "lab", "weight"]):
    p, _ = R.bias_free(g)
    m = metrics(p.y, p.pred)
    rows.append({"variant": variant, "label source": lab, "weight": weight, "MAE": m["MAE"], "pearson": m["pearson"]})
pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True).round(3)

## 4. Ridge, Random Forest and Gradient Boosting, tuned, compared without selection bias

In [ ]:
choice, table, pools, picks = R.decide(cfg, tuned, base, d, RES)
table.round(3)

## 5. The rule for keeping a complex model

In [ ]:
rule = choice["rule"]
print(f"best Ridge-type model : {rule['ridge']}  (MAE {choice['ridge_best']['MAE']:.3f})")
print(f"best tree model       : {rule['complex']}  (MAE {choice['complex_best']['MAE']:.3f})")
print(f"gain {rule['gain_mae']:+.3f} MAE, 95% interval {rule['ci'][0]:+.3f} to {rule['ci'][1]:+.3f}, better in {rule['folds_better']} of 5 folds")
chosen_name = choice["complex_best"]["model"] if rule["passes_1_3"] else choice["ridge_best"]["model"]
chosen_variant = choice["complex_best"]["variant"] if rule["passes_1_3"] else choice["ridge_best"]["variant"]
ridge_name, ridge_variant = choice["ridge_best"]["model"], choice["ridge_best"]["variant"]
print("model after rules 1-3:", chosen_name, "|", chosen_variant)

In [ ]:
json.dump({**choice, "chosen": [chosen_name, chosen_variant]}, open(RES / "model_choice.json", "w"), indent=1, default=float)
print("model for the final test:", chosen_name, "|", chosen_variant)

## 6. Final test, scored once

In [ ]:
final, fits = R.final_test(cfg, d, chosen_name, chosen_variant, tuned, settings)
final.to_csv(RES / "final_test_predictions_unguarded.csv", index=False)
MAIN = f"{chosen_name} | {chosen_variant}"

fold_info = pd.DataFrame({o: fits[(chosen_name, chosen_variant, o)][2] for o in GROUPS}).T
display(fold_info)
print(len(final[final.name == MAIN]), "test predictions from", len(GROUPS), "models, each made by a model that never saw that student")

In [ ]:
crit, m = R.criteria_table(cfg, final, MAIN, d)
crit.to_csv(RES / "final_test_criteria.csv", index=False)
print("PRIMARY: all %d real records" % len(final[final.name == MAIN]))
display(crit.round(3))
f20 = final[final.record_id.isin(P20)]
crit20, m20 = R.criteria_table(cfg, f20, MAIN, d)
crit20.to_csv(RES / "final_test_criteria_original20.csv", index=False)
print("SECONDARY: the original 20 records only (%d)" % len(f20[f20.name == MAIN]))
display(crit20.round(3))

In [ ]:
rows = []
for name, g in final.groupby("name"):
    mm = metrics(g.y, g.pred)
    rows.append({"model": name, "MAE": mm["MAE"], "pearson": mm["pearson"], "spearman": mm["spearman"], "bias": mm["bias"]})
display(pd.DataFrame(rows).set_index("model").sort_values("MAE").round(3))
pf = final[final.name == MAIN].assign(err=lambda x: (x.pred - x.y).abs())
display(pf.groupby("outer_test").agg(n=("err", "size"), MAE=("err", "mean")).round(3).T)
f = final[final.name == MAIN]
fig, ax = plt.subplots(figsize=(4.8, 4.4))
ax.scatter(f.y, f.pred, c=np.where(f.activity > 0, "#2b6cb0", "#c53030"), s=24)
ax.plot([0, 1], [0, 1], color="grey", lw=1)
ax.set_xlabel("rubric rating"); ax.set_ylabel("predicted"); ax.set_title(MAIN, fontsize=8)
fig.tight_layout(); plt.show()

## 7. Beside the first clean run and the uncleaned experiments

In [ ]:
print("First clean run (labels: 20 ratings + estimates), criteria table:")
display(pd.read_csv(Path("../clean/training/results/final_test_criteria.csv")).round(3))
print("Uncleaned experiments (old features, 20 ratings), criteria table:")
display(pd.read_csv(Path("../training/results/final_test_criteria.csv")).round(3))

## 8. Reading the results

_Reading of the results: written after the run._